# Step 3 — Rolling-Window De-Anchoring

Slides a 63-day window across the sample to test whether the UK's idiosyncratic share spiked around the May-2026 political shocks.

**Script notes**

```
Step 3 - Rolling-window de-anchoring
Does the UK's idiosyncratic (non-global) share spike around May-2026 political shocks,
even though it is LOW on full-sample average (step 2)?

For each rolling window:
  - PCA on correlation matrix of daily changes (7 countries, 10y)
  - record: PC1 variance share; UK communality on PC1 (=UK R^2 on global factor);
            UK idiosyncratic share (1 - communality); UK-PC1 loading;
            avg pairwise corr of UK with the other 6; UK-DE corr; UK-US corr
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import numpy as np
import pandas as pd

In [2]:
ch = pd.read_csv("./changes_bp.csv", index_col=0, parse_dates=True)
CTRY10 = ["US_10y", "UK_10y", "DE_10y", "FR_10y", "IT_10y", "JP_10y", "CA_10y"]
labels = [c[:2] for c in CTRY10]
uk = labels.index("UK")
X = ch[CTRY10]

In [3]:
WIN = 63  # ~3 trading months

In [4]:
def pca_window(Zw):
    C = np.corrcoef(Zw, rowvar=False)
    ev, evec = np.linalg.eigh(C)
    o = np.argsort(ev)[::-1]
    ev, evec = ev[o], evec[:, o]
    load = evec * np.sqrt(ev)
    if (load[:, 0] < 0).sum() > len(labels)/2:
        load[:, 0] *= -1
    return ev/ev.sum(), load, C

In [5]:
rows = []
for end in range(WIN, len(X)+1):
    w = X.iloc[end-WIN:end]
    Zw = ((w - w.mean())/w.std(ddof=1)).values
    vshare, load, C = pca_window(Zw)
    uk_comm = load[uk, 0]**2
    uk_other_corr = np.mean([C[uk, j] for j in range(len(labels)) if j != uk])
    rows.append({
        "date": w.index[-1],
        "PC1_var": vshare[0],
        "UK_PC1_loading": load[uk, 0],
        "UK_common_PC1": uk_comm,
        "UK_idiosyncratic": 1 - uk_comm,
        "UK_avg_corr_others": uk_other_corr,
        "UK_DE_corr": C[uk, labels.index("DE")],
        "UK_US_corr": C[uk, labels.index("US")],
    })
roll = pd.DataFrame(rows).set_index("date")

In [6]:
print("="*66)
print(f"ROLLING-WINDOW DE-ANCHORING  (window={WIN} trading days, 10y panel)")
print(f"windows: {roll.index.min().date()} -> {roll.index.max().date()}  ({len(roll)})")
print("="*66)

ROLLING-WINDOW DE-ANCHORING  (window=63 trading days, 10y panel)
windows: 2024-04-12 -> 2026-05-29  (457)


In [7]:
# article event dates
events = {
    "2024-10-30": "Reeves first Budget",
    "2026-02-26": "US-Iran war begins (approx)",
    "2026-05-08": "Local elections / Starmer clings on",
    "2026-05-15": "Burnham challenge / gilts worst day; Big Read",
}

In [8]:
print("\nUK idiosyncratic share around key episodes:")
fullsample = 0.283  # from step 2
print(f"  (full-sample UK idiosyncratic = {fullsample:.3f})")
for d, lab in events.items():
    dt = pd.Timestamp(d)
    nearest = roll.index[roll.index.get_indexer([dt], method='nearest')[0]]
    r = roll.loc[nearest]
    print(f"  {nearest.date()} [{lab:38s}] idio={r['UK_idiosyncratic']:.3f}  "
          f"PC1load={r['UK_PC1_loading']:.3f}  UK-DE corr={r['UK_DE_corr']:.3f}")


UK idiosyncratic share around key episodes:
  (full-sample UK idiosyncratic = 0.283)
  2024-10-30 [Reeves first Budget                   ] idio=0.255  PC1load=0.863  UK-DE corr=0.484
  2026-02-26 [US-Iran war begins (approx)           ] idio=0.335  PC1load=0.816  UK-DE corr=0.623
  2026-05-08 [Local elections / Starmer clings on   ] idio=0.174  PC1load=0.909  UK-DE corr=0.529
  2026-05-15 [Burnham challenge / gilts worst day; Big Read] idio=0.153  PC1load=0.920  UK-DE corr=0.542


In [9]:
print("\nUK idiosyncratic share - distribution over full rolling sample:")
q = roll["UK_idiosyncratic"].describe(percentiles=[.1,.25,.5,.75,.9])
print(q.round(3).to_string())


UK idiosyncratic share - distribution over full rolling sample:
count    457.000
mean       0.277
std        0.075
min        0.153
10%        0.183
25%        0.222
50%        0.267
75%        0.320
90%        0.402
max        0.469


In [10]:
# peak idiosyncratic windows
print("\nTop 5 windows by UK idiosyncratic share:")
top = roll.sort_values("UK_idiosyncratic", ascending=False).head(5)
print(top[["UK_idiosyncratic","UK_PC1_loading","UK_DE_corr","UK_US_corr"]].round(3).to_string())


Top 5 windows by UK idiosyncratic share:
            UK_idiosyncratic  UK_PC1_loading  UK_DE_corr  UK_US_corr
date                                                                
2025-08-05             0.469           0.729       0.170       0.602
2025-06-24             0.445           0.745       0.143       0.647
2025-07-14             0.442           0.747       0.126       0.647
2025-06-11             0.441           0.747       0.291       0.574
2025-07-30             0.432           0.754       0.091       0.629


In [11]:
# was the May-2026 window elevated vs its own history (percentile)?
may = roll.loc["2026-05-01":"2026-05-29", "UK_idiosyncratic"]
if len(may):
    pct = (roll["UK_idiosyncratic"] < may.max()).mean()*100
    print(f"\nMay-2026 peak UK idiosyncratic = {may.max():.3f} "
          f"({pct:.0f}th percentile of all windows)")


May-2026 peak UK idiosyncratic = 0.198 (18th percentile of all windows)


In [12]:
roll.to_csv("./pca10_rolling.csv")
print("\nsaved pca10_rolling.csv")


saved pca10_rolling.csv
